# Notebook 05 - Fusion Strategies

In [1]:
# Core imports
import os
import re
import gc
import json
import time
import random
import shutil
from pathlib import Path

# Reduce TensorFlow log noise before importing TensorFlow.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Disable XLA/JIT because repeated fold training can otherwise trigger slow compilation.
tf.config.optimizer.set_jit(False)

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("XLA/JIT enabled:", tf.config.optimizer.get_jit())


2026-06-28 07:38:45.171262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782632325.384015      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782632325.456514      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782632325.965424      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782632325.965475      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782632325.965478      22 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
XLA/JIT enabled: 


In [ ]:
# Fixed experiment configuration
WINDOW_SIZE = 30
STEP_SIZE = 10
TARGET_COLUMN = "Energy expenditures (W/kg)"
TIME_COLUMN = "Time (s)"
ACTIVITY_COLUMN = "Activity Code"
SUBJECT_COLUMN = "Subject"
WKG_TO_KCAL_PER_MIN = 60 / 4184

FINAL_MODEL_CONFIG = {
    "name": "baseline_tiny_residual_tcn",
    "filters": 32,
    "kernel_size": 3,
    "dilation_pattern": [1, 2, 4],
    "dense_units": 32,
    "dropout": 0.10,
    "l2_strength": 0.0,
    "learning_rate": 0.001,
    "batch_size": 64,
    "max_epochs": 100,
    "early_stopping_patience": 12,
}

SUBJECT_WEIGHTS_KG = {
    1: 63.49, 2: 63.49, 3: 71.20, 4: 68.03, 5: 68.03,
    6: 68.03, 7: 95.24, 8: 65.76, 9: 68.93, 10: 58.05,
}

ALL_INPUT_SIGNALS = [
    "Waist Acceleration",
    "Chest Acceleration",
    "Left Ankle Acceleration",
    "right Ankle Acceleration",
    "left wrist Acceleration",
    "right wrist Acceleration",
    "EMG_magnitude_left",
    "EMG_magnitude_right",
    "left wrist electrodermal",
    "right wrist electrodermal",
    "left wrist Temperature",
    "right wrist Temperature",
    "Breath Frequency",
    "Minute Ventilation",
    "SpO2",
    "Heart Rate",
]

ACCELERATION_SIGNALS = [
    "Waist Acceleration",
    "Chest Acceleration",
    "Left Ankle Acceleration",
    "right Ankle Acceleration",
    "left wrist Acceleration",
    "right wrist Acceleration",
]

SIGNAL_GROUPS = {
    "G1": {
        "name": "Local",
        "signals": ACCELERATION_SIGNALS + ["EMG_magnitude_left", "EMG_magnitude_right"],
        "note": "All accelerations plus both EMG magnitude signals.",
    },
    "G2": {
        "name": "Global",
        "signals": [
            "left wrist electrodermal",
            "right wrist electrodermal",
            "left wrist Temperature",
            "right wrist Temperature",
            "Breath Frequency",
            "Minute Ventilation",
            "Heart Rate",
            "SpO2",
        ],
        "note": "EDA, skin temperature, respiratory and pulse/oxygenation signals.",
    },
    "G3": {
        "name": "Hexoskin",
        "signals": ["Waist Acceleration", "Breath Frequency", "Minute Ventilation", "Heart Rate"],
        "note": "PDF lists 5 signals, but only 4 are identifiable here; Waist Acceleration is used as hip/waist proxy.",
    },
    "G4": {
        "name": "Physiological",
        "signals": ["Breath Frequency", "Minute Ventilation", "Heart Rate", "SpO2"],
        "note": "Physiological signal group from the project description.",
    },
    "G5": {
        "name": "Motion",
        "signals": ACCELERATION_SIGNALS,
        "note": "All acceleration signals only.",
    },
    "G6": {
        "name": "Local + Global",
        "signals": ALL_INPUT_SIGNALS,
        "note": "All 16 allowed signals, excluding VO2 and target leakage columns.",
    },
}

FUSION_STRATEGIES = ["early_fusion", "intermediate_fusion", "late_fusion"]

# -----------------------------------------------------------------------------
# Run selection
# -----------------------------------------------------------------------------
# Edit this block depending on what still needs to be trained.
#
# Examples:
#   GROUPS_TO_RUN = ["G1", "G2", "G3"]
#   GROUPS_TO_RUN = ["G4", "G5", "G6"]
#   GROUPS_TO_RUN = list(SIGNAL_GROUPS.keys())
#
GROUPS_TO_RUN = ["G6"]
STRATEGIES_TO_RUN = FUSION_STRATEGIES

MAX_NEW_FOLDS = 200

FORCE_RERUN = False
MANUAL_DATA_DIR = None

AUTO_RESUME_FROM_KAGGLE_INPUT = True
MANUAL_RESUME_DIR = None

START_FROM_SCRATCH_IF_NO_RESUME = True

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/phase_d_fusion_all_groups")
else:
    OUTPUT_DIR = Path("phase_d_fusion_all_groups_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOLD_RESULTS_PATH = OUTPUT_DIR / "phase_d_fusion_fold_results.csv"
HISTORY_PATH = OUTPUT_DIR / "phase_d_fusion_learning_curves.csv"
VALIDATION_SEGMENTS_PATH = OUTPUT_DIR / "phase_d_fusion_validation_segment_predictions.csv"
TEST_SEGMENTS_PATH = OUTPUT_DIR / "phase_d_fusion_test_segment_predictions.csv"

# Known final best single-signal baselines from Notebook 04. These are used for quick comparison.
EMBEDDED_GROUP_BASELINES = {
    "G1": {"Best Single Signal": "right Ankle Acceleration", "RMSE_kcal_min": 2.052},
    "G2": {"Best Single Signal": "Minute Ventilation", "RMSE_kcal_min": 0.991},
    "G3": {"Best Single Signal": "Minute Ventilation", "RMSE_kcal_min": 0.991},
    "G4": {"Best Single Signal": "Minute Ventilation", "RMSE_kcal_min": 0.991},
    "G5": {"Best Single Signal": "right Ankle Acceleration", "RMSE_kcal_min": 2.052},
    "G6": {"Best Single Signal": "Minute Ventilation", "RMSE_kcal_min": 0.991},
}

for group_id, group_info in SIGNAL_GROUPS.items():
    missing = [signal for signal in group_info["signals"] if signal not in ALL_INPUT_SIGNALS]
    if missing:
        raise ValueError(f"{group_id} contains signals not in ALL_INPUT_SIGNALS: {missing}")

display(pd.DataFrame([
    {
        "Group": group_id,
        "Name": group_info["name"],
        "Number of Signals": len(group_info["signals"]),
        "Signals": ", ".join(group_info["signals"]),
        "Note": group_info["note"],
    }
    for group_id, group_info in SIGNAL_GROUPS.items()
]))


,Group,Name,Number of Signals,Signals,Note
0,G1,Local,8,"Waist Acceleration, Chest Acceleration, Left A...",All accelerations plus both EMG magnitude sign...
1,G2,Global,8,"left wrist electrodermal, right wrist electrod...","EDA, skin temperature, respiratory and pulse/o..."
2,G3,Hexoskin,4,"Waist Acceleration, Breath Frequency, Minute V...","PDF lists 5 signals, but only 4 are identifiab..."
3,G4,Physiological,4,"Breath Frequency, Minute Ventilation, Heart Ra...",Physiological signal group from the project de...
4,G5,Motion,6,"Waist Acceleration, Chest Acceleration, Left A...",All acceleration signals only.
5,G6,Local + Global,16,"Waist Acceleration, Chest Acceleration, Left A...","All 16 allowed signals, excluding VO2 and targ..."


## Resume previous Kaggle outputs or run a group subset

In [3]:
RESUME_FILE_NAMES = [
    "phase_d_fusion_fold_results.csv",
    "phase_d_fusion_learning_curves.csv",
    "phase_d_fusion_validation_segment_predictions.csv",
    "phase_d_fusion_test_segment_predictions.csv",
]


def count_completed_rows(results_path):
    """Count completed fold rows in a candidate resume file."""
    try:
        return len(pd.read_csv(results_path, usecols=["Group", "Strategy", "Fold"]))
    except Exception:
        return -1


def list_resume_candidates():
    """List all visible Phase-D fold-result files in Kaggle inputs and manual resume dirs."""
    candidates = []

    if MANUAL_RESUME_DIR is not None:
        manual_dir = Path(MANUAL_RESUME_DIR)
        candidates.append(manual_dir / "phase_d_fusion_fold_results.csv")

    if AUTO_RESUME_FROM_KAGGLE_INPUT and Path("/kaggle/input").exists():
        candidates.extend(Path("/kaggle/input").rglob("phase_d_fusion_fold_results.csv"))

    rows = []
    for candidate in sorted(set(candidates), key=lambda path: str(path)):
        rows.append({
            "Candidate File": str(candidate),
            "Exists": candidate.exists(),
            "Completed Rows": count_completed_rows(candidate) if candidate.exists() else -1,
            "Parent Directory": str(candidate.parent),
        })
    return pd.DataFrame(rows)


def find_best_resume_dir():
    """Find the candidate directory with the largest existing fold-results file."""
    candidates = list_resume_candidates()
    if candidates.empty:
        return None
    candidates = candidates[candidates["Completed Rows"] >= 0].copy()
    if candidates.empty:
        return None
    candidates = candidates.sort_values("Completed Rows", ascending=False)
    return Path(candidates.iloc[0]["Parent Directory"])


def copy_resume_outputs_if_needed():
    """Copy previous fold CSVs into OUTPUT_DIR unless current files already exist."""
    current_rows = count_completed_rows(FOLD_RESULTS_PATH) if FOLD_RESULTS_PATH.exists() else 0
    if current_rows > 0:
        print("Current OUTPUT_DIR already contains completed fold rows:", current_rows)
        return

    candidates = list_resume_candidates()
    print("Resume candidate files visible to this notebook:")
    if candidates.empty:
        print("No candidate files found under /kaggle/input.")
    else:
        display(candidates)

    resume_dir = find_best_resume_dir()
    if resume_dir is None:
        message = (
            "No previous Phase-D CSV output was found. Kaggle will start from scratch unless "
            "START_FROM_SCRATCH_IF_NO_RESUME is set to True. Add the previous run's output files "
            "as an input dataset, or upload a dataset/folder that contains phase_d_fusion_fold_results.csv."
        )
        if not START_FROM_SCRATCH_IF_NO_RESUME:
            raise RuntimeError(message)
        print(message)
        print("START_FROM_SCRATCH_IF_NO_RESUME=True, so continuing with a fresh run.")
        return

    if resume_dir.resolve() == OUTPUT_DIR.resolve():
        print("Resume directory is already the current output directory:", resume_dir)
        return

    copied = []
    skipped = []
    for file_name in RESUME_FILE_NAMES:
        source = resume_dir / file_name
        target = OUTPUT_DIR / file_name
        if not source.exists():
            continue
        if target.exists():
            skipped.append(file_name)
            continue
        shutil.copy2(source, target)
        copied.append(file_name)

    rows = count_completed_rows(OUTPUT_DIR / "phase_d_fusion_fold_results.csv")
    print("Best resume directory:", resume_dir)
    print("Copied resume files:", copied if copied else "none")
    print("Skipped existing files:", skipped if skipped else "none")
    print("Completed fold rows available before training:", rows)


copy_resume_outputs_if_needed()


Resume candidate files visible to this notebook:
No candidate files found under /kaggle/input.
No previous Phase-D CSV output was found. Kaggle will start from scratch unless START_FROM_SCRATCH_IF_NO_RESUME is set to True. Add the previous run's output files as an input dataset, or upload a dataset/folder that contains phase_d_fusion_fold_results.csv.
START_FROM_SCRATCH_IF_NO_RESUME=True, so continuing with a fresh run.


## Data loading, segmentation, and window construction


In [ ]:
def parse_subject_id(path):
    """Extract subject id from a file name like Subject (1).csv."""
    match = re.search(r"Subject \((\d+)\)", path.name)
    if match is None:
        raise ValueError(f"Could not parse subject id from {path.name}")
    return int(match.group(1))


def find_data_dir():
    """Find the folder containing the subject CSV files."""
    candidates = []
    if MANUAL_DATA_DIR is not None:
        candidates.append(Path(MANUAL_DATA_DIR))
    candidates.extend([
        Path("/kaggle/input"),
        Path("/kaggle/input/data-20260607"),
        Path("/kaggle/input/projekt-2"),
        Path("/kaggle/input/projekt2"),
        Path("Data-20260607"),
        Path("Projekt 2/Data-20260607"),
    ])
    for candidate in candidates:
        if not candidate.exists():
            continue
        direct_matches = sorted(candidate.glob("Subject (*.csv"))
        if direct_matches:
            return candidate
        recursive_matches = sorted(candidate.rglob("Subject (*.csv"))
        if recursive_matches:
            return recursive_matches[0].parent
    raise FileNotFoundError("Could not find Subject CSV files. Set MANUAL_DATA_DIR manually.")


def load_dataset(data_dir):
    """Load all subject files and preserve their original row order."""
    frames = []
    for file_path in sorted(data_dir.glob("Subject (*.csv"), key=parse_subject_id):
        subject_id = parse_subject_id(file_path)
        frame = pd.read_csv(file_path)
        frame.columns = [column.strip() for column in frame.columns]
        frame["Original_Row_Order"] = np.arange(len(frame))
        frame[SUBJECT_COLUMN] = subject_id
        frame["Weight_kg"] = SUBJECT_WEIGHTS_KG[subject_id]
        frame["Target_kcal_min"] = frame[TARGET_COLUMN] * frame["Weight_kg"] * WKG_TO_KCAL_PER_MIN
        frames.append(frame)
    data = pd.concat(frames, ignore_index=True)
    return data.sort_values([SUBJECT_COLUMN, "Original_Row_Order"]).reset_index(drop=True)


def validate_columns(data):
    """Validate that all required columns exist."""
    required = [TIME_COLUMN, SUBJECT_COLUMN, ACTIVITY_COLUMN, TARGET_COLUMN] + ALL_INPUT_SIGNALS
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def add_segment_ids(data):
    """Detect continuous trials using original order, activity changes, and time continuity."""
    data = data.sort_values([SUBJECT_COLUMN, "Original_Row_Order"]).reset_index(drop=True).copy()
    previous_subject = data[SUBJECT_COLUMN].shift(1)
    previous_activity = data[ACTIVITY_COLUMN].shift(1)
    previous_time = data[TIME_COLUMN].shift(1)
    time_difference = data[TIME_COLUMN] - previous_time

    new_segment = (
        (data[SUBJECT_COLUMN] != previous_subject)
        | (data[ACTIVITY_COLUMN] != previous_activity)
        | (((time_difference < 0) | (time_difference > 1.5)).fillna(True))
    )
    data["Segment ID"] = new_segment.cumsum().astype(int)
    return data


def create_windows(data, signals, window_size=30, step_size=10):
    """Create overlapping windows within each continuous segment."""
    windows = []
    rows = []
    window_id = 0

    for segment_id, segment in data.groupby("Segment ID", sort=True):
        segment = segment.sort_values("Original_Row_Order").reset_index(drop=True)
        if len(segment) < window_size:
            continue

        for start_index in range(0, len(segment) - window_size + 1, step_size):
            end_index = start_index + window_size
            windows.append(segment.loc[start_index:end_index - 1, signals].to_numpy(dtype=np.float32))
            rows.append({
                "Window ID": window_id,
                "Segment ID": int(segment_id),
                "Subject": int(segment[SUBJECT_COLUMN].iloc[0]),
                "Activity Code": int(segment[ACTIVITY_COLUMN].iloc[0]),
                "Start Index": int(start_index),
                "End Index": int(end_index),
                "Target_W_kg": float(segment[TARGET_COLUMN].iloc[0]),
                "Weight_kg": float(segment["Weight_kg"].iloc[0]),
                "Target_kcal_min": float(segment["Target_kcal_min"].iloc[0]),
            })
            window_id += 1

    X = np.stack(windows).astype(np.float32)
    metadata = pd.DataFrame(rows)
    y_wkg = metadata["Target_W_kg"].to_numpy(dtype=np.float32)
    return X, y_wkg, metadata


DATA_DIR = find_data_dir()
raw_data = load_dataset(DATA_DIR)
validate_columns(raw_data)
data = add_segment_ids(raw_data)
X_all, y_wkg, metadata = create_windows(data, ALL_INPUT_SIGNALS, WINDOW_SIZE, STEP_SIZE)
SIGNAL_TO_INDEX = {signal: index for index, signal in enumerate(ALL_INPUT_SIGNALS)}

print("Data directory:", DATA_DIR)
print("Raw data shape:", raw_data.shape)
print("Number of detected continuous segments:", data["Segment ID"].nunique())
print("All-signal window tensor shape:", X_all.shape)
print("Metadata shape:", metadata.shape)
display(metadata.head())

assert data["Segment ID"].nunique() == 198, "Expected 198 continuous trials."
assert X_all.shape == (6724, 30, 16), "Expected all-signal window shape (6724, 30, 16)."


## Two-stage LOSO, normalization, and metrics

For every test subject:

1. **Epoch selection:** 8 train subjects + 1 validation subject. Early stopping selects the best epoch count.
2. **Final training:** a fresh model is trained for exactly that epoch count on all 9 non-test subjects.
3. **Testing:** the held-out subject is evaluated once.

Normalization is fit only on the training subjects of the respective stage. Missing values are set to 0 after standardization, which means "training mean" in standardized space.


In [5]:
def create_loso_splits(subjects):
    """Create deterministic two-stage LOSO splits."""
    subjects = [int(subject) for subject in sorted(subjects)]
    splits = []
    for fold_index, test_subject in enumerate(subjects):
        validation_subject = subjects[(fold_index + 1) % len(subjects)]
        epoch_train_subjects = [s for s in subjects if s not in [test_subject, validation_subject]]
        final_train_subjects = [s for s in subjects if s != test_subject]
        splits.append({
            "Fold": fold_index + 1,
            "Epoch Selection Train Subjects": epoch_train_subjects,
            "Validation Subject": validation_subject,
            "Final Train Subjects": final_train_subjects,
            "Test Subject": test_subject,
        })
    return splits


def get_epoch_selection_indices(metadata, split):
    train_mask = metadata["Subject"].isin(split["Epoch Selection Train Subjects"]).to_numpy()
    validation_mask = (metadata["Subject"] == split["Validation Subject"]).to_numpy()
    return np.where(train_mask)[0], np.where(validation_mask)[0]


def get_final_training_indices(metadata, split):
    train_mask = metadata["Subject"].isin(split["Final Train Subjects"]).to_numpy()
    test_mask = (metadata["Subject"] == split["Test Subject"]).to_numpy()
    return np.where(train_mask)[0], np.where(test_mask)[0]


def get_group_array(group_id):
    """Select the channel subset for one signal group."""
    group_signals = SIGNAL_GROUPS[group_id]["signals"]
    channel_indices = [SIGNAL_TO_INDEX[signal] for signal in group_signals]
    return X_all[:, :, channel_indices]


def normalize_by_training_data(X_train, *other_arrays):
    """Standardize each channel using training data only and then impute missing values with 0."""
    mean = np.nanmean(X_train, axis=(0, 1), keepdims=True)
    std = np.nanstd(X_train, axis=(0, 1), keepdims=True)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std = np.where((std < 1e-8) | np.isnan(std), 1.0, std)

    def transform(array):
        normalized = (array - mean) / std
        return np.nan_to_num(normalized, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    return (transform(X_train),) + tuple(transform(array) for array in other_arrays)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred, prefix, unit):
    return {
        f"{prefix}_RMSE_{unit}": rmse(y_true, y_pred),
        f"{prefix}_MAE_{unit}": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}_R2_{unit}": float(r2_score(y_true, y_pred)),
    }


def aggregate_to_segments(frame):
    """Aggregate overlapping window predictions back to trial/segment level."""
    return (
        frame.groupby(["Segment ID", "Subject", "Activity Code"], as_index=False)
        .agg(
            Actual_W_kg=("Actual_W_kg", "first"),
            Predicted_W_kg=("Predicted_W_kg", "mean"),
            Actual_kcal_min=("Actual_kcal_min", "first"),
            Predicted_kcal_min=("Predicted_kcal_min", "mean"),
            Weight_kg=("Weight_kg", "first"),
        )
    )


def evaluate_predictions(metadata_subset, y_true_wkg, y_pred_wkg, prefix):
    """Compute window-level and segment-level metrics in W/kg and kcal/min."""
    frame = metadata_subset.copy().reset_index(drop=True)
    weights = frame["Weight_kg"].to_numpy(dtype=np.float32)
    frame["Actual_W_kg"] = y_true_wkg
    frame["Predicted_W_kg"] = y_pred_wkg
    frame["Actual_kcal_min"] = y_true_wkg * weights * WKG_TO_KCAL_PER_MIN
    frame["Predicted_kcal_min"] = y_pred_wkg * weights * WKG_TO_KCAL_PER_MIN
    segments = aggregate_to_segments(frame)

    metrics = {}
    metrics.update(regression_metrics(frame["Actual_W_kg"], frame["Predicted_W_kg"], f"{prefix}_Window", "W_kg"))
    metrics.update(regression_metrics(frame["Actual_kcal_min"], frame["Predicted_kcal_min"], f"{prefix}_Window", "kcal_min"))
    metrics.update(regression_metrics(segments["Actual_W_kg"], segments["Predicted_W_kg"], f"{prefix}_Segment", "W_kg"))
    metrics.update(regression_metrics(segments["Actual_kcal_min"], segments["Predicted_kcal_min"], f"{prefix}_Segment", "kcal_min"))
    return metrics, segments


loso_splits = create_loso_splits(sorted(metadata["Subject"].unique()))
display(pd.DataFrame(loso_splits))

for split in loso_splits:
    epoch_train = set(split["Epoch Selection Train Subjects"])
    validation = {split["Validation Subject"]}
    final_train = set(split["Final Train Subjects"])
    test = {split["Test Subject"]}
    assert epoch_train.isdisjoint(validation)
    assert final_train == epoch_train.union(validation)
    assert final_train.isdisjoint(test)
    assert len(final_train) == 9

print("All two-stage LOSO splits passed leakage validation.")


,Fold,Epoch Selection Train Subjects,Validation Subject,Final Train Subjects,Test Subject
0,1,"[3, 4, 5, 6, 7, 8, 9, 10]",2,"[2, 3, 4, 5, 6, 7, 8, 9, 10]",1
1,2,"[1, 4, 5, 6, 7, 8, 9, 10]",3,"[1, 3, 4, 5, 6, 7, 8, 9, 10]",2
2,3,"[1, 2, 5, 6, 7, 8, 9, 10]",4,"[1, 2, 4, 5, 6, 7, 8, 9, 10]",3
3,4,"[1, 2, 3, 6, 7, 8, 9, 10]",5,"[1, 2, 3, 5, 6, 7, 8, 9, 10]",4
4,5,"[1, 2, 3, 4, 7, 8, 9, 10]",6,"[1, 2, 3, 4, 6, 7, 8, 9, 10]",5
5,6,"[1, 2, 3, 4, 5, 8, 9, 10]",7,"[1, 2, 3, 4, 5, 7, 8, 9, 10]",6
6,7,"[1, 2, 3, 4, 5, 6, 9, 10]",8,"[1, 2, 3, 4, 5, 6, 8, 9, 10]",7
7,8,"[1, 2, 3, 4, 5, 6, 7, 10]",9,"[1, 2, 3, 4, 5, 6, 7, 9, 10]",8
8,9,"[1, 2, 3, 4, 5, 6, 7, 8]",10,"[1, 2, 3, 4, 5, 6, 7, 8, 10]",9
9,10,"[2, 3, 4, 5, 6, 7, 8, 9]",1,"[1, 2, 3, 4, 5, 6, 7, 8, 9]",10


All two-stage LOSO splits passed leakage validation.


## Fusion architectures

All models use the same frozen Tiny Residual TCN building block.

- Early fusion has one multichannel TCN.
- Intermediate fusion has one TCN encoder per signal, then concatenates embeddings.
- Late fusion has one TCN prediction branch per signal, then combines the predictions with a learned linear layer.


In [6]:
def safe_name(text):
    """Create a compact Keras-safe layer-name fragment."""
    return re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()


def compile_model(model):
    """Compile a regression model with the frozen optimizer settings."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=FINAL_MODEL_CONFIG["learning_rate"]),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae"), keras.metrics.RootMeanSquaredError(name="rmse")],
        jit_compile=False,
    )
    return model


def get_regularizer():
    """Return L2 regularizer if enabled in the frozen config."""
    strength = FINAL_MODEL_CONFIG.get("l2_strength", 0.0)
    return keras.regularizers.l2(strength) if strength and strength > 0 else None


def residual_tcn_block(x, block_prefix, dilation_rate):
    """Build one residual TCN block."""
    filters = FINAL_MODEL_CONFIG["filters"]
    kernel_size = FINAL_MODEL_CONFIG["kernel_size"]
    dropout = FINAL_MODEL_CONFIG["dropout"]
    regularizer = get_regularizer()
    shortcut = x

    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu",
        kernel_regularizer=regularizer,
        name=f"{block_prefix}_conv1",
    )(x)
    x = layers.BatchNormalization(name=f"{block_prefix}_bn1")(x)
    x = layers.Dropout(dropout, name=f"{block_prefix}_dropout")(x)
    x = layers.Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        kernel_regularizer=regularizer,
        name=f"{block_prefix}_conv2",
    )(x)
    x = layers.BatchNormalization(name=f"{block_prefix}_bn2")(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(
            filters,
            1,
            padding="same",
            kernel_regularizer=regularizer,
            name=f"{block_prefix}_shortcut",
        )(shortcut)

    x = layers.Add(name=f"{block_prefix}_add")([shortcut, x])
    return layers.Activation("relu", name=f"{block_prefix}_relu")(x)


def build_tcn_embedding(input_tensor, branch_name):
    """Encode one signal or one multichannel input into a dense embedding."""
    x = input_tensor
    for dilation_rate in FINAL_MODEL_CONFIG["dilation_pattern"]:
        x = residual_tcn_block(x, f"{branch_name}_d{dilation_rate}", dilation_rate)
    x = layers.GlobalAveragePooling1D(name=f"{branch_name}_gap")(x)
    x = layers.Dense(
        FINAL_MODEL_CONFIG["dense_units"],
        activation="relu",
        kernel_regularizer=get_regularizer(),
        name=f"{branch_name}_embedding",
    )(x)
    x = layers.Dropout(FINAL_MODEL_CONFIG["dropout"], name=f"{branch_name}_embedding_dropout")(x)
    return x


def build_early_fusion_model(group_id, group_signals):
    """Fuse all channels before the first TCN layer."""
    group_prefix = safe_name(group_id)
    inputs = keras.Input(shape=(WINDOW_SIZE, len(group_signals)), name=f"{group_prefix}_multichannel_input")
    embedding = build_tcn_embedding(inputs, f"{group_prefix}_early")
    output = layers.Dense(1, kernel_regularizer=get_regularizer(), name="ee_output")(embedding)
    return compile_model(keras.Model(inputs, output, name=f"{group_prefix}_early_fusion"))


def build_intermediate_fusion_model(group_id, group_signals):
    """Encode each signal separately, then concatenate learned embeddings."""
    group_prefix = safe_name(group_id)
    inputs = []
    embeddings = []

    for signal_index, signal_name in enumerate(group_signals):
        signal_prefix = f"{group_prefix}_i{signal_index}_{safe_name(signal_name)}"
        signal_input = keras.Input(shape=(WINDOW_SIZE, 1), name=f"{signal_prefix}_input")
        signal_embedding = build_tcn_embedding(signal_input, signal_prefix)
        inputs.append(signal_input)
        embeddings.append(signal_embedding)

    fused = layers.Concatenate(name=f"{group_prefix}_intermediate_concat")(embeddings)
    fused = layers.Dense(
        FINAL_MODEL_CONFIG["dense_units"],
        activation="relu",
        kernel_regularizer=get_regularizer(),
        name=f"{group_prefix}_intermediate_dense",
    )(fused)
    fused = layers.Dropout(FINAL_MODEL_CONFIG["dropout"], name=f"{group_prefix}_intermediate_dropout")(fused)
    output = layers.Dense(1, kernel_regularizer=get_regularizer(), name="ee_output")(fused)
    return compile_model(keras.Model(inputs, output, name=f"{group_prefix}_intermediate_fusion"))


def build_late_fusion_model(group_id, group_signals):
    """Produce one prediction per signal and combine them with a trainable linear layer."""
    group_prefix = safe_name(group_id)
    inputs = []
    signal_predictions = []

    for signal_index, signal_name in enumerate(group_signals):
        signal_prefix = f"{group_prefix}_l{signal_index}_{safe_name(signal_name)}"
        signal_input = keras.Input(shape=(WINDOW_SIZE, 1), name=f"{signal_prefix}_input")
        signal_embedding = build_tcn_embedding(signal_input, signal_prefix)
        signal_prediction = layers.Dense(
            1,
            kernel_regularizer=get_regularizer(),
            name=f"{signal_prefix}_ee_prediction",
        )(signal_embedding)
        inputs.append(signal_input)
        signal_predictions.append(signal_prediction)

    prediction_vector = layers.Concatenate(name=f"{group_prefix}_late_prediction_vector")(signal_predictions)
    output = layers.Dense(
        1,
        use_bias=True,
        kernel_initializer=keras.initializers.Constant(1.0 / len(group_signals)),
        bias_initializer="zeros",
        name="late_fusion_combiner",
    )(prediction_vector)
    return compile_model(keras.Model(inputs, output, name=f"{group_prefix}_late_fusion"))


def build_fusion_model(group_id, strategy):
    """Build one of the three required fusion models for one signal group."""
    group_signals = SIGNAL_GROUPS[group_id]["signals"]
    if strategy == "early_fusion":
        return build_early_fusion_model(group_id, group_signals)
    if strategy == "intermediate_fusion":
        return build_intermediate_fusion_model(group_id, group_signals)
    if strategy == "late_fusion":
        return build_late_fusion_model(group_id, group_signals)
    raise ValueError(f"Unknown fusion strategy: {strategy}")


def prepare_model_inputs(X_group, strategy):
    """Convert a multichannel tensor into the input format required by a strategy."""
    if strategy == "early_fusion":
        return X_group
    return [X_group[:, :, [channel_index]] for channel_index in range(X_group.shape[-1])]


architecture_summary = []
for group_id in SIGNAL_GROUPS:
    for strategy in FUSION_STRATEGIES:
        model = build_fusion_model(group_id, strategy)
        architecture_summary.append({
            "Group": group_id,
            "Group Name": SIGNAL_GROUPS[group_id]["name"],
            "Strategy": strategy,
            "Number of Signals": len(SIGNAL_GROUPS[group_id]["signals"]),
            "Number of Inputs": len(model.inputs),
            "Number of Parameters": int(model.count_params()),
        })
        del model
        tf.keras.backend.clear_session()
        gc.collect()

architecture_summary = pd.DataFrame(architecture_summary)
display(architecture_summary)
architecture_summary.to_csv(OUTPUT_DIR / "phase_d_architecture_summary.csv", index=False)


I0000 00:00:1782632344.627962      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782632344.634218      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


,Group,Group Name,Strategy,Number of Signals,Number of Inputs,Number of Parameters
0,G1,Local,early_fusion,8,1,18465
1,G1,Local,intermediate_fusion,8,8,148545
2,G1,Local,late_fusion,8,8,140561
3,G2,Global,early_fusion,8,1,18465
4,G2,Global,intermediate_fusion,8,8,148545
5,G2,Global,late_fusion,8,8,140561
6,G3,Hexoskin,early_fusion,4,1,17953
7,G3,Hexoskin,intermediate_fusion,4,4,74305
8,G3,Hexoskin,late_fusion,4,4,70281
9,G4,Physiological,early_fusion,4,1,17953


## Two-stage training helpers with resume logic

In [7]:
def predict_in_batches(model, model_inputs, batch_size=512):
    """Predict in batches for array or list-of-array model inputs."""
    total_rows = len(model_inputs[0]) if isinstance(model_inputs, list) else len(model_inputs)
    predictions = []

    for start in range(0, total_rows, batch_size):
        end = start + batch_size
        if isinstance(model_inputs, list):
            batch = [tf.convert_to_tensor(array[start:end], dtype=tf.float32) for array in model_inputs]
        else:
            batch = tf.convert_to_tensor(model_inputs[start:end], dtype=tf.float32)
        predictions.append(model(batch, training=False).numpy().reshape(-1))

    return np.concatenate(predictions)


def load_existing_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def append_dataframe_to_csv(frame, path):
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def fold_already_done(existing_results, group_id, strategy, fold):
    if existing_results.empty:
        return False
    return bool(
        (
            (existing_results["Group"] == group_id)
            & (existing_results["Strategy"] == strategy)
            & (existing_results["Fold"] == fold)
        ).any()
    )


def reset_fold_seed(group_index, strategy_index, fold, stage):
    """Use deterministic but distinct seeds for each group, strategy, fold, and stage."""
    stage_offset = 0 if stage == "epoch_selection" else 100000
    fold_seed = SEED + group_index * 10000 + strategy_index * 1000 + fold * 10 + stage_offset
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    tf.keras.utils.set_random_seed(fold_seed)
    return fold_seed


def extract_late_fusion_weights(model):
    """Return learned linear combiner weights and bias for late fusion."""
    layer = model.get_layer("late_fusion_combiner")
    kernel, bias = layer.get_weights()
    return kernel.reshape(-1), float(bias[0])


def make_result_base(group_id, strategy, split):
    """Create a result row with stable columns across all groups."""
    group_info = SIGNAL_GROUPS[group_id]
    row = {
        "Group": group_id,
        "Group Name": group_info["name"],
        "Group Note": group_info["note"],
        "Number of Signals": len(group_info["signals"]),
        "Signals": json.dumps(group_info["signals"]),
        "Strategy": strategy,
        "Protocol": "two_stage_8_train_1_validation_then_9_train_1_test",
        "Fold": int(split["Fold"]),
        "Epoch Selection Train Subjects": json.dumps([int(s) for s in split["Epoch Selection Train Subjects"]]),
        "Validation Subject": int(split["Validation Subject"]),
        "Final Train Subjects": json.dumps([int(s) for s in split["Final Train Subjects"]]),
        "Test Subject": int(split["Test Subject"]),
    }
    for signal in ALL_INPUT_SIGNALS:
        row[f"Late Weight - {signal}"] = np.nan
    row["Late Fusion Bias"] = np.nan
    return row


def train_fusion_fold_two_stage(group_id, strategy, group_index, strategy_index, split):
    """Run final two-stage LOSO training for one group, strategy, and fold."""
    group_signals = SIGNAL_GROUPS[group_id]["signals"]
    X_group_all = get_group_array(group_id)

    # Stage 1: epoch selection.
    tf.keras.backend.clear_session()
    gc.collect()
    stage1_seed = reset_fold_seed(group_index, strategy_index, split["Fold"], "epoch_selection")

    epoch_train_idx, validation_idx = get_epoch_selection_indices(metadata, split)
    X_epoch_train, X_validation = normalize_by_training_data(X_group_all[epoch_train_idx], X_group_all[validation_idx])
    y_epoch_train = y_wkg[epoch_train_idx]
    y_validation = y_wkg[validation_idx]

    selection_model = build_fusion_model(group_id, strategy)
    parameter_count = int(selection_model.count_params())
    selection_train_inputs = prepare_model_inputs(X_epoch_train, strategy)
    validation_inputs = prepare_model_inputs(X_validation, strategy)

    selection_start = time.time()
    selection_history = selection_model.fit(
        selection_train_inputs,
        y_epoch_train,
        validation_data=(validation_inputs, y_validation),
        epochs=FINAL_MODEL_CONFIG["max_epochs"],
        batch_size=FINAL_MODEL_CONFIG["batch_size"],
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=FINAL_MODEL_CONFIG["early_stopping_patience"],
            restore_best_weights=True,
            min_delta=1e-4,
        )],
        verbose=0,
    )
    selection_time = time.time() - selection_start
    best_epoch = int(np.argmin(selection_history.history["val_loss"]) + 1)

    validation_pred = predict_in_batches(selection_model, validation_inputs)
    validation_metrics, validation_segments = evaluate_predictions(
        metadata.iloc[validation_idx], y_validation, validation_pred, "Validation"
    )

    selection_history_frame = pd.DataFrame(selection_history.history)
    selection_history_frame["Group"] = group_id
    selection_history_frame["Group Name"] = SIGNAL_GROUPS[group_id]["name"]
    selection_history_frame["Strategy"] = strategy
    selection_history_frame["Fold"] = int(split["Fold"])
    selection_history_frame["Stage"] = "epoch_selection"
    selection_history_frame["Epoch"] = np.arange(1, len(selection_history_frame) + 1)

    del selection_model
    tf.keras.backend.clear_session()
    gc.collect()

    # Stage 2: fresh final model trained on all nine non-test subjects.
    stage2_seed = reset_fold_seed(group_index, strategy_index, split["Fold"], "final_training")
    final_train_idx, test_idx = get_final_training_indices(metadata, split)
    X_final_train, X_test = normalize_by_training_data(X_group_all[final_train_idx], X_group_all[test_idx])
    y_final_train = y_wkg[final_train_idx]
    y_test = y_wkg[test_idx]

    final_model = build_fusion_model(group_id, strategy)
    final_train_inputs = prepare_model_inputs(X_final_train, strategy)
    test_inputs = prepare_model_inputs(X_test, strategy)

    final_start = time.time()
    final_history = final_model.fit(
        final_train_inputs,
        y_final_train,
        epochs=best_epoch,
        batch_size=FINAL_MODEL_CONFIG["batch_size"],
        verbose=0,
    )
    final_time = time.time() - final_start

    test_pred = predict_in_batches(final_model, test_inputs)
    test_metrics, test_segments = evaluate_predictions(metadata.iloc[test_idx], y_test, test_pred, "Test")

    result = make_result_base(group_id, strategy, split)
    result.update({
        "Best Epoch": best_epoch,
        "Epoch Selection Trained Epochs": len(selection_history.history["loss"]),
        "Epoch Selection Time Seconds": selection_time,
        "Final Training Time Seconds": final_time,
        "Total Training Time Seconds": selection_time + final_time,
        "Number of Parameters": parameter_count,
        "Epoch Selection Seed": stage1_seed,
        "Final Training Seed": stage2_seed,
    })

    if strategy == "late_fusion":
        late_weights, late_bias = extract_late_fusion_weights(final_model)
        result["Late Fusion Bias"] = late_bias
        for signal_index, signal_name in enumerate(group_signals):
            result[f"Late Weight - {signal_name}"] = float(late_weights[signal_index])

    result.update(validation_metrics)
    result.update(test_metrics)

    final_history_frame = pd.DataFrame(final_history.history)
    final_history_frame["Group"] = group_id
    final_history_frame["Group Name"] = SIGNAL_GROUPS[group_id]["name"]
    final_history_frame["Strategy"] = strategy
    final_history_frame["Fold"] = int(split["Fold"])
    final_history_frame["Stage"] = "final_training_9_subjects"
    final_history_frame["Epoch"] = np.arange(1, len(final_history_frame) + 1)

    validation_segments["Group"] = group_id
    validation_segments["Group Name"] = SIGNAL_GROUPS[group_id]["name"]
    validation_segments["Strategy"] = strategy
    validation_segments["Fold"] = int(split["Fold"])
    validation_segments["Stage"] = "epoch_selection"

    test_segments["Group"] = group_id
    test_segments["Group Name"] = SIGNAL_GROUPS[group_id]["name"]
    test_segments["Strategy"] = strategy
    test_segments["Fold"] = int(split["Fold"])
    test_segments["Stage"] = "final_model_9_subjects"

    del final_model
    tf.keras.backend.clear_session()
    gc.collect()

    combined_history = pd.concat([selection_history_frame, final_history_frame], ignore_index=True, sort=False)
    return result, combined_history, validation_segments, test_segments


## Run Phase D

In [8]:
if FORCE_RERUN:
    for path in [FOLD_RESULTS_PATH, HISTORY_PATH, VALIDATION_SEGMENTS_PATH, TEST_SEGMENTS_PATH]:
        if path.exists():
            path.unlink()

existing_results = load_existing_csv(FOLD_RESULTS_PATH)
new_folds_completed = 0
stop_requested = False

for group_index, group_id in enumerate(GROUPS_TO_RUN):
    group_info = SIGNAL_GROUPS[group_id]
    print("=" * 110)
    print(f"Group {group_id}: {group_info['name']} | {len(group_info['signals'])} signals")
    print("Signals:", ", ".join(group_info["signals"]))
    print("Note:", group_info["note"])

    for strategy_index, strategy in enumerate(STRATEGIES_TO_RUN):
        print("-" * 110)
        print("Fusion strategy:", strategy)

        for split in loso_splits:
            if MAX_NEW_FOLDS is not None and new_folds_completed >= MAX_NEW_FOLDS:
                stop_requested = True
                break

            if not FORCE_RERUN and fold_already_done(existing_results, group_id, strategy, split["Fold"]):
                print(f"Skipping existing result: {group_id} | {strategy} | fold {split['Fold']}")
                continue

            print(
                f"Training {group_id} | {strategy} | fold {split['Fold']} | "
                f"validation subject {split['Validation Subject']} | test subject {split['Test Subject']}"
            )

            result, history_frame, validation_segments, test_segments = train_fusion_fold_two_stage(
                group_id, strategy, group_index, strategy_index, split
            )

            append_dataframe_to_csv(pd.DataFrame([result]), FOLD_RESULTS_PATH)
            append_dataframe_to_csv(history_frame, HISTORY_PATH)
            append_dataframe_to_csv(validation_segments, VALIDATION_SEGMENTS_PATH)
            append_dataframe_to_csv(test_segments, TEST_SEGMENTS_PATH)
            existing_results = load_existing_csv(FOLD_RESULTS_PATH)
            new_folds_completed += 1

            print(
                "Validation Segment RMSE:", round(result["Validation_Segment_RMSE_kcal_min"], 3), "kcal/min",
                "| Test Segment RMSE:", round(result["Test_Segment_RMSE_kcal_min"], 3), "kcal/min",
                "| Best epoch:", result["Best Epoch"],
                "| Parameters:", result["Number of Parameters"],
            )

        if stop_requested:
            break
    if stop_requested:
        break

print("Finished requested Phase-D fusion chunk.")
print("New folds completed in this run:", new_folds_completed)
print("Results path:", FOLD_RESULTS_PATH)


Group G6: Local + Global | 16 signals
Signals: Waist Acceleration, Chest Acceleration, Left Ankle Acceleration, right Ankle Acceleration, left wrist Acceleration, right wrist Acceleration, EMG_magnitude_left, EMG_magnitude_right, left wrist electrodermal, right wrist electrodermal, left wrist Temperature, right wrist Temperature, Breath Frequency, Minute Ventilation, SpO2, Heart Rate
Note: All 16 allowed signals, excluding VO2 and target leakage columns.
--------------------------------------------------------------------------------------------------------------
Fusion strategy: early_fusion
Training G6 | early_fusion | fold 1 | validation subject 2 | test subject 1


I0000 00:00:1782632372.824892      69 cuda_dnn.cc:529] Loaded cuDNN version 91002


Validation Segment RMSE: 0.809 kcal/min | Test Segment RMSE: 1.554 kcal/min | Best epoch: 8 | Parameters: 19489
Training G6 | early_fusion | fold 2 | validation subject 3 | test subject 2
Validation Segment RMSE: 1.122 kcal/min | Test Segment RMSE: 1.011 kcal/min | Best epoch: 5 | Parameters: 19489
Training G6 | early_fusion | fold 3 | validation subject 4 | test subject 3
Validation Segment RMSE: 0.71 kcal/min | Test Segment RMSE: 1.117 kcal/min | Best epoch: 14 | Parameters: 19489
Training G6 | early_fusion | fold 4 | validation subject 5 | test subject 4
Validation Segment RMSE: 0.695 kcal/min | Test Segment RMSE: 0.788 kcal/min | Best epoch: 30 | Parameters: 19489
Training G6 | early_fusion | fold 5 | validation subject 6 | test subject 5
Validation Segment RMSE: 0.7 kcal/min | Test Segment RMSE: 0.958 kcal/min | Best epoch: 10 | Parameters: 19489
Training G6 | early_fusion | fold 6 | validation subject 7 | test subject 6
Validation Segment RMSE: 1.72 kcal/min | Test Segment RMSE: 

## Ranking and comparison

In [9]:
fusion_fold_results = load_existing_csv(FOLD_RESULTS_PATH)
fusion_histories = load_existing_csv(HISTORY_PATH)
fusion_validation_segments = load_existing_csv(VALIDATION_SEGMENTS_PATH)
fusion_test_segments = load_existing_csv(TEST_SEGMENTS_PATH)

print("Fold results shape:", fusion_fold_results.shape)
print("Learning curves shape:", fusion_histories.shape)
print("Validation segment predictions shape:", fusion_validation_segments.shape)
print("Test segment predictions shape:", fusion_test_segments.shape)

expected_folds = len(SIGNAL_GROUPS) * len(FUSION_STRATEGIES) * len(loso_splits)
print("Expected full Phase-D folds:", expected_folds)
print("Completed folds:", len(fusion_fold_results))

if fusion_fold_results.empty:
    print("No fold results available yet. Run the training cell first.")
else:
    completion_table = (
        fusion_fold_results.groupby(["Group", "Group Name", "Strategy"])
        .agg(Completed_Folds=("Fold", "nunique"))
        .reset_index()
        .sort_values(["Group", "Strategy"])
    )
    display(completion_table)

    group_strategy_ranking = (
        fusion_fold_results.groupby(["Group", "Group Name", "Strategy"])
        .agg(
            Completed_Folds=("Fold", "nunique"),
            Mean_Test_Segment_RMSE_kcal_min=("Test_Segment_RMSE_kcal_min", "mean"),
            Std_Test_Segment_RMSE_kcal_min=("Test_Segment_RMSE_kcal_min", "std"),
            Mean_Test_Segment_MAE_kcal_min=("Test_Segment_MAE_kcal_min", "mean"),
            Mean_Test_Segment_R2=("Test_Segment_R2_kcal_min", "mean"),
            Mean_Test_Segment_RMSE_W_kg=("Test_Segment_RMSE_W_kg", "mean"),
            Mean_Validation_Segment_RMSE_kcal_min=("Validation_Segment_RMSE_kcal_min", "mean"),
            Mean_Best_Epoch=("Best Epoch", "mean"),
            Mean_Parameters=("Number of Parameters", "mean"),
            Mean_Total_Training_Time_Seconds=("Total Training Time Seconds", "mean"),
        )
        .reset_index()
        .sort_values(["Group", "Mean_Test_Segment_RMSE_kcal_min"])
    )

    group_baselines = pd.DataFrame([
        {
            "Group": group_id,
            "Best Single Signal": baseline["Best Single Signal"],
            "Best_Single_RMSE_kcal_min": baseline["RMSE_kcal_min"],
        }
        for group_id, baseline in EMBEDDED_GROUP_BASELINES.items()
    ])

    group_strategy_ranking = group_strategy_ranking.merge(group_baselines, on="Group", how="left")
    group_strategy_ranking["Improvement_vs_Best_Single_Percent"] = (
        (group_strategy_ranking["Best_Single_RMSE_kcal_min"] - group_strategy_ranking["Mean_Test_Segment_RMSE_kcal_min"])
        / group_strategy_ranking["Best_Single_RMSE_kcal_min"]
        * 100
    )

    best_strategy_per_group = (
        group_strategy_ranking.sort_values(["Group", "Mean_Test_Segment_RMSE_kcal_min"])
        .groupby("Group", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    overall_ranking = group_strategy_ranking.sort_values("Mean_Test_Segment_RMSE_kcal_min").reset_index(drop=True)

    group_strategy_ranking.to_csv(OUTPUT_DIR / "phase_d_group_strategy_ranking.csv", index=False)
    best_strategy_per_group.to_csv(OUTPUT_DIR / "phase_d_best_strategy_per_group.csv", index=False)
    overall_ranking.to_csv(OUTPUT_DIR / "phase_d_overall_ranking.csv", index=False)
    fusion_fold_results.to_csv(OUTPUT_DIR / "phase_d_fusion_fold_results_export.csv", index=False)
    fusion_histories.to_csv(OUTPUT_DIR / "phase_d_fusion_learning_curves_export.csv", index=False)
    fusion_test_segments.to_csv(OUTPUT_DIR / "phase_d_fusion_test_segment_predictions_export.csv", index=False)

    print("Group-strategy ranking:")
    display(group_strategy_ranking)
    print("Best strategy per group:")
    display(best_strategy_per_group)
    print("Overall ranking:")
    display(overall_ranking.head(20))


Fold results shape: (30, 61)
Learning curves shape: (1386, 12)
Validation segment predictions shape: (594, 13)
Test segment predictions shape: (594, 13)
Expected full Phase-D folds: 180
Completed folds: 30


,Group,Group Name,Strategy,Completed_Folds
0,G6,Local + Global,early_fusion,10
1,G6,Local + Global,intermediate_fusion,10
2,G6,Local + Global,late_fusion,10


Group-strategy ranking:


,Group,Group Name,Strategy,Completed_Folds,Mean_Test_Segment_RMSE_kcal_min,Std_Test_Segment_RMSE_kcal_min,Mean_Test_Segment_MAE_kcal_min,Mean_Test_Segment_R2,Mean_Test_Segment_RMSE_W_kg,Mean_Validation_Segment_RMSE_kcal_min,Mean_Best_Epoch,Mean_Parameters,Mean_Total_Training_Time_Seconds,Best Single Signal,Best_Single_RMSE_kcal_min,Improvement_vs_Best_Single_Percent
0,G6,Local + Global,intermediate_fusion,10,1.075271,0.317288,0.851974,0.865581,1.072010,0.703855,21.0,297025.0,937.379157,Minute Ventilation,0.991,-8.503666
1,G6,Local + Global,early_fusion,10,1.246465,0.471076,0.993039,0.768761,1.253414,0.977084,9.8,19489.0,55.858629,Minute Ventilation,0.991,-25.778556
2,G6,Local + Global,late_fusion,10,1.331990,0.683306,1.063394,0.773605,1.308873,0.780430,20.5,281121.0,969.330572,Minute Ventilation,0.991,-34.408695


Best strategy per group:


,Group,Group Name,Strategy,Completed_Folds,Mean_Test_Segment_RMSE_kcal_min,Std_Test_Segment_RMSE_kcal_min,Mean_Test_Segment_MAE_kcal_min,Mean_Test_Segment_R2,Mean_Test_Segment_RMSE_W_kg,Mean_Validation_Segment_RMSE_kcal_min,Mean_Best_Epoch,Mean_Parameters,Mean_Total_Training_Time_Seconds,Best Single Signal,Best_Single_RMSE_kcal_min,Improvement_vs_Best_Single_Percent
0,G6,Local + Global,intermediate_fusion,10,1.075271,0.317288,0.851974,0.865581,1.07201,0.703855,21.0,297025.0,937.379157,Minute Ventilation,0.991,-8.503666


Overall ranking:


,Group,Group Name,Strategy,Completed_Folds,Mean_Test_Segment_RMSE_kcal_min,Std_Test_Segment_RMSE_kcal_min,Mean_Test_Segment_MAE_kcal_min,Mean_Test_Segment_R2,Mean_Test_Segment_RMSE_W_kg,Mean_Validation_Segment_RMSE_kcal_min,Mean_Best_Epoch,Mean_Parameters,Mean_Total_Training_Time_Seconds,Best Single Signal,Best_Single_RMSE_kcal_min,Improvement_vs_Best_Single_Percent
0,G6,Local + Global,intermediate_fusion,10,1.075271,0.317288,0.851974,0.865581,1.072010,0.703855,21.0,297025.0,937.379157,Minute Ventilation,0.991,-8.503666
1,G6,Local + Global,early_fusion,10,1.246465,0.471076,0.993039,0.768761,1.253414,0.977084,9.8,19489.0,55.858629,Minute Ventilation,0.991,-25.778556
2,G6,Local + Global,late_fusion,10,1.331990,0.683306,1.063394,0.773605,1.308873,0.780430,20.5,281121.0,969.330572,Minute Ventilation,0.991,-34.408695


## Visual diagnostics

In [ ]:
if not fusion_fold_results.empty:
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=group_strategy_ranking,
        x="Group",
        y="Mean_Test_Segment_RMSE_kcal_min",
        hue="Strategy",
        order=sorted(SIGNAL_GROUPS.keys()),
    )
    plt.title("Phase D: Fusion Performance by Signal Group")
    plt.xlabel("Signal Group")
    plt.ylabel("Mean Test Segment RMSE (kcal/min)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "phase_d_group_strategy_barplot.png", dpi=160)
    plt.show()

    heatmap = group_strategy_ranking.pivot_table(
        index="Group",
        columns="Strategy",
        values="Mean_Test_Segment_RMSE_kcal_min",
    ).loc[sorted(SIGNAL_GROUPS.keys())]
    plt.figure(figsize=(9, 5))
    sns.heatmap(heatmap, annot=True, fmt=".2f", cmap="viridis_r")
    plt.title("Phase D: Mean Test Segment RMSE (kcal/min)")
    plt.xlabel("Fusion Strategy")
    plt.ylabel("Signal Group")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "phase_d_group_strategy_heatmap.png", dpi=160)
    plt.show()

    plt.figure(figsize=(13, 6))
    sns.boxplot(
        data=fusion_fold_results,
        x="Group",
        y="Test_Segment_RMSE_kcal_min",
        hue="Strategy",
        order=sorted(SIGNAL_GROUPS.keys()),
    )
    plt.title("Phase D: Fold-to-Fold Variation Across Test Subjects")
    plt.xlabel("Signal Group")
    plt.ylabel("Test Segment RMSE (kcal/min)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "phase_d_fold_variation_boxplot.png", dpi=160)
    plt.show()


In [ ]:
def plot_learning_curves_for_model(group_id, strategy, folds_to_plot=(1, 5, 10)):
    """Plot epoch-selection learning curves for selected folds."""
    subset = fusion_histories[
        (fusion_histories["Group"] == group_id)
        & (fusion_histories["Strategy"] == strategy)
        & (fusion_histories["Stage"] == "epoch_selection")
    ]
    if subset.empty:
        print(f"No learning curves found for {group_id} / {strategy}.")
        return

    plt.figure(figsize=(11, 5))
    for fold in folds_to_plot:
        fold_history = subset[subset["Fold"] == fold]
        if fold_history.empty:
            continue
        plt.plot(fold_history["Epoch"], fold_history["loss"], linestyle="--", alpha=0.55, label=f"Fold {fold} train")
        plt.plot(fold_history["Epoch"], fold_history["val_loss"], linewidth=2, alpha=0.85, label=f"Fold {fold} val")
    plt.title(f"Learning Curves: {group_id} / {strategy}")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"learning_curves_{group_id}_{strategy}.png", dpi=160)
    plt.show()


if not fusion_fold_results.empty:
    for _, row in best_strategy_per_group.iterrows():
        plot_learning_curves_for_model(row["Group"], row["Strategy"])


In [ ]:
def plot_predicted_vs_actual(group_id, strategy):
    """Plot segment-level predictions against ground truth for one group/strategy."""
    subset = fusion_test_segments[
        (fusion_test_segments["Group"] == group_id)
        & (fusion_test_segments["Strategy"] == strategy)
    ].copy()
    if subset.empty:
        print(f"No test segment predictions found for {group_id} / {strategy}.")
        return

    plt.figure(figsize=(7, 7))
    sns.scatterplot(
        data=subset,
        x="Actual_W_kg",
        y="Predicted_W_kg",
        hue="Subject",
        palette="tab10",
        s=55,
    )
    min_value = min(subset["Actual_W_kg"].min(), subset["Predicted_W_kg"].min())
    max_value = max(subset["Actual_W_kg"].max(), subset["Predicted_W_kg"].max())
    plt.plot([min_value, max_value], [min_value, max_value], "k--", label="Perfect Prediction")
    plt.title(f"Predicted vs Actual EE: {group_id} / {strategy}")
    plt.xlabel("Actual EE (W/kg)")
    plt.ylabel("Predicted EE (W/kg)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"predicted_vs_actual_{group_id}_{strategy}.png", dpi=160)
    plt.show()


if not fusion_fold_results.empty:
    for _, row in best_strategy_per_group.iterrows():
        plot_predicted_vs_actual(row["Group"], row["Strategy"])


In [ ]:
if not fusion_fold_results.empty:
    late_weight_rows = []
    late_results = fusion_fold_results[fusion_fold_results["Strategy"] == "late_fusion"]

    for group_id, group_info in SIGNAL_GROUPS.items():
        group_late = late_results[late_results["Group"] == group_id]
        if group_late.empty:
            continue
        for signal in group_info["signals"]:
            column = f"Late Weight - {signal}"
            late_weight_rows.append({
                "Group": group_id,
                "Group Name": group_info["name"],
                "Signal": signal,
                "Mean Weight": group_late[column].mean(),
                "Std Weight": group_late[column].std(),
                "Mean Absolute Weight": group_late[column].abs().mean(),
            })

    late_weight_summary = pd.DataFrame(late_weight_rows)
    if not late_weight_summary.empty:
        late_weight_summary = late_weight_summary.sort_values(["Group", "Mean Absolute Weight"], ascending=[True, False])
        late_weight_summary.to_csv(OUTPUT_DIR / "phase_d_late_fusion_weight_summary.csv", index=False)
        display(late_weight_summary)

        for group_id in sorted(late_weight_summary["Group"].unique()):
            subset = late_weight_summary[late_weight_summary["Group"] == group_id]
            plt.figure(figsize=(8, max(4, 0.35 * len(subset))))
            sns.barplot(data=subset, x="Mean Weight", y="Signal", order=subset["Signal"])
            plt.axvline(0, color="black", linewidth=1)
            plt.title(f"Late Fusion Weights: {group_id}")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f"late_fusion_weights_{group_id}.png", dpi=160)
            plt.show()


## Final export and interpretation helper

In [ ]:
if fusion_fold_results.empty:
    print("No final interpretation written because no results are available yet.")
else:
    best_overall = overall_ranking.iloc[0]

    interpretation = f"""
Notebook 06 ran Phase-D fusion experiments with the frozen baseline_tiny_residual_tcn architecture.

Completed folds: {len(fusion_fold_results)} / {expected_folds}

Best overall model so far:
Group: {best_overall['Group']} ({best_overall['Group Name']})
Strategy: {best_overall['Strategy']}
Mean Test Segment RMSE: {best_overall['Mean_Test_Segment_RMSE_kcal_min']:.3f} kcal/min
Std Test Segment RMSE: {best_overall['Std_Test_Segment_RMSE_kcal_min']:.3f} kcal/min
Mean Test Segment MAE: {best_overall['Mean_Test_Segment_MAE_kcal_min']:.3f} kcal/min
Mean Test Segment R2: {best_overall['Mean_Test_Segment_R2']:.3f}
Mean parameter count: {best_overall['Mean_Parameters']:.0f}
Mean total training time per fold: {best_overall['Mean_Total_Training_Time_Seconds']:.1f} seconds
"""

    print(interpretation)
    with open(OUTPUT_DIR / "phase_d_fusion_interpretation.txt", "w") as file:
        file.write(interpretation)

    with open(OUTPUT_DIR / "phase_d_fusion_config.json", "w") as file:
        json.dump({
            "notebook": "06-phase-d-fusion-all-groups-two-stage-loso.ipynb",
            "groups": SIGNAL_GROUPS,
            "strategies": FUSION_STRATEGIES,
            "groups_to_run": GROUPS_TO_RUN,
            "strategies_to_run": STRATEGIES_TO_RUN,
            "g3_assumption": "G3 uses Waist Acceleration + Breath Frequency + Minute Ventilation + Heart Rate; this gives 4 identifiable signals although the PDF lists 5.",
            "intermediate_encoder_weights": "separate per signal",
            "late_fusion_combination": "trainable linear layer over per-signal predictions",
            "base_model_config": FINAL_MODEL_CONFIG,
            "window_size": WINDOW_SIZE,
            "step_size": STEP_SIZE,
            "protocol": "two-stage LOSO: 8 train + 1 validation for epoch selection, then fresh training on all 9 non-test subjects",
        }, file, indent=2)

    print("Generated files:")
    for path in sorted(OUTPUT_DIR.glob("*")):
        print("-", path)
